# **Fundación YPF — EnergIA Digital | Proyecto Final Integrador**
## **Aprendizaje No Supervisado: Clustering de Siniestros Viales en CABA**

**Grupo 8:** Diego Bianqui, Luida Medina y Matías Lorenzo  
**Dataset:** registros oficiales de siniestros viales de CABA (2019–2025), Observatorio de Movilidad y Seguridad Vial (OMSV).

### Propósito
Identificar tipologías de siniestros a partir de variables temporales, territoriales y de interacción entre actores viales. El análisis compara **MiniBatch K-Means** y **DBSCAN**, explicita las decisiones metodológicas y perfila los grupos obtenidos.

> **Regla contra data leakage:** `gravedad_siniestro`, los conteos de víctimas y cualquier desenlace médico se excluyen de la matriz de clustering. La gravedad se utiliza únicamente después de asignar los grupos, para describirlos y comparar su composición.

## **1. Preparación del entorno y carga de datos**

Se utilizan las mismas fuentes locales del proyecto. El notebook puede ejecutarse desde la raíz del repositorio y no requiere rutas absolutas ni descargas externas.

In [ ]:
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shapely.geometry import Point, shape
from shapely.prepared import prep

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import MiniBatchKMeans, DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

PATH_HECHOS = 'data/siniestros_viales_hechos.csv'
PATH_COMUNAS = 'data/comunas.json'

df_raw = pd.read_csv(PATH_HECHOS, sep=';', low_memory=False)
print(f'Dataset cargado: {df_raw.shape[0]:,} siniestros y {df_raw.shape[1]} columnas')
display(df_raw.head())

## **2. Limpieza e ingeniería de variables contextuales**

Se derivan hora, día de semana, franja horaria, condición de fin de semana y hora pico. La comuna se extrae del campo original y, cuando falta y existen coordenadas válidas, se intenta imputar mediante *Point-in-Polygon* con los polígonos oficiales de CABA. Se normalizan las variables categóricas y se construyen indicadores de presencia de actores viales.

Los indicadores de actores se derivan de los textos de participantes, modo de desplazamiento y contraparte; no se utilizan variables que describan el resultado médico.

In [ ]:
df = df_raw.copy()

# Fecha y hora
fecha = pd.to_datetime(df.get('fecha_siniestro'), dayfirst=True, errors='coerce')
df['hora_num'] = pd.to_numeric(
    df.get('hora_siniestro', pd.Series(index=df.index, dtype='object'))
      .astype('string').str.extract(r'(\d{1,2})', expand=False),
    errors='coerce'
)
df['mes_num'] = fecha.dt.month
df['dia_semana_num'] = fecha.dt.dayofweek
dias_es = {0:'Lunes',1:'Martes',2:'Miércoles',3:'Jueves',4:'Viernes',5:'Sábado',6:'Domingo'}
df['dia_semana_nombre'] = df['dia_semana_num'].map(dias_es)
df['es_fin_de_semana'] = df['dia_semana_num'].isin([5,6]).astype('int8')

# Franja horaria y hora pico
bins = [-0.1, 5.99, 11.99, 17.99, 24]
labels = ['Madrugada', 'Mañana', 'Tarde', 'Noche']
df['franja_horaria'] = pd.cut(df['hora_num'], bins=bins, labels=labels, include_lowest=True)
df['es_hora_pico'] = (
    df['dia_semana_num'].between(0,4) &
    (df['hora_num'].between(7,9) | df['hora_num'].between(17,19))
).astype('int8')

# Coordenadas: parseo numérico defensivo
for col in ['longitud_siniestro', 'latitud_siniestro']:
    if col not in df.columns:
        df[col] = np.nan
    else:
        texto = df[col].astype('string').str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(texto.str.extract(r'(-?\d+(?:\.\d+)?)', expand=False), errors='coerce')

# Comuna desde el campo original
if 'comuna_siniestro' in df.columns:
    df['comuna_num'] = pd.to_numeric(
        df['comuna_siniestro'].astype('string').str.extract(r'(\d{1,2})', expand=False),
        errors='coerce'
    )
else:
    df['comuna_num'] = np.nan
df.loc[~df['comuna_num'].between(1,15), 'comuna_num'] = np.nan

# Imputación geoespacial de comuna para coordenadas disponibles
with open(PATH_COMUNAS, 'r', encoding='utf-8') as f:
    geojson = json.load(f)
poligonos = []
for feature in geojson['features']:
    comuna = int(feature['properties']['comuna'])
    geom = prep(shape(feature['geometry']))
    poligonos.append((comuna, geom))

def buscar_comuna(lon, lat):
    if pd.isna(lon) or pd.isna(lat):
        return np.nan
    punto = Point(float(lon), float(lat))
    for numero, poligono in poligonos:
        if poligono.contains(punto):
            return numero
    return np.nan

mask_comuna = df['comuna_num'].isna() & df['longitud_siniestro'].notna() & df['latitud_siniestro'].notna()
if mask_comuna.any():
    df.loc[mask_comuna, 'comuna_num'] = df.loc[mask_comuna].apply(
        lambda row: buscar_comuna(row['longitud_siniestro'], row['latitud_siniestro']), axis=1
    )

zonas = {
    1:'Centro',3:'Centro',5:'Centro',6:'Centro',7:'Centro',15:'Centro',
    2:'Norte',12:'Norte',13:'Norte',14:'Norte',
    4:'Sur',8:'Sur',9:'Sur',10:'Oeste',11:'Oeste'
}
df['zona_caba'] = df['comuna_num'].map(zonas)

# Normalización de variables categóricas y participantes
for col in ['tipo_de_via_siniestro','modo_desplazamiento_victima','contraparte_siniestro','participantes_siniestro']:
    if col not in df.columns:
        df[col] = 'Sin datos'
    df[col] = df[col].astype('string').str.upper().str.strip()
    df[col] = df[col].replace(['', 'SD', 'S/D', 'SIN DATO', 'NAN', 'NONE', '<NA>'], pd.NA)

texto_actores = (
    df['participantes_siniestro'].fillna('') + ' ' +
    df['modo_desplazamiento_victima'].fillna('') + ' ' +
    df['contraparte_siniestro'].fillna('')
).str.upper()

df['involucra_moto'] = texto_actores.str.contains(r'MOTO|MOTOCIC', regex=True).astype('int8')
df['involucra_peaton'] = texto_actores.str.contains(r'PEAT[ÓO]N|PEATON', regex=True).astype('int8')
df['involucra_bicicleta'] = texto_actores.str.contains(r'BICI|CICLIST', regex=True).astype('int8')
df['involucra_transporte_publico'] = texto_actores.str.contains(r'COLECTIV|AUTOB[ÚU]S|BUS', regex=True).astype('int8')
df['involucra_auto'] = texto_actores.str.contains(r'AUTO|AUTOM[ÓO]VIL', regex=True).astype('int8')
df['involucra_pesado'] = texto_actores.str.contains(r'CAMI[ÓO]N|CAMION|T[ÓO]RRIDO|TREN', regex=True).astype('int8')
df['involucra_vulnerable'] = df[['involucra_moto','involucra_peaton','involucra_bicicleta']].max(axis=1).astype('int8')

# Variables de análisis (la gravedad queda reservada para el perfil posterior)
variables_categoricas = [
    'tipo_de_via_siniestro','franja_horaria','dia_semana_nombre',
    'zona_caba','modo_desplazamiento_victima','contraparte_siniestro'
]
variables_numericas = [
    'hora_num','mes_num','dia_semana_num','comuna_num',
    'longitud_siniestro','latitud_siniestro'
]
variables_binarias = [
    'es_fin_de_semana','es_hora_pico','involucra_moto','involucra_peaton',
    'involucra_bicicleta','involucra_transporte_publico','involucra_auto',
    'involucra_pesado','involucra_vulnerable'
]

for col in variables_categoricas:
    df[col] = df[col].fillna('Sin datos').astype(str)

X_cluster = df[variables_categoricas + variables_numericas + variables_binarias].copy()
assert not any('gravedad' in c.lower() or 'victim' in c.lower() for c in X_cluster.columns)
print('Variables para clustering:', X_cluster.shape[1])
print('Registros:', len(X_cluster))
print('Comunas identificadas:', int(df['comuna_num'].notna().sum()))
print('Variables excluidas por diseño: gravedad y desenlaces/conteos de víctimas.')

## **3. Preprocesamiento para variables mixtas**

Las variables numéricas se imputan por mediana y se escalan. Las categóricas se imputan con una categoría explícita y se codifican con *One-Hot Encoding*. Las variables binarias se imputan por moda. Se conserva una representación dispersa para controlar el consumo de memoria.

In [ ]:
preprocessor_cluster = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), variables_categoricas),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=False))
        ]), variables_numericas),
        ('bin', SimpleImputer(strategy='most_frequent'), variables_binarias)
    ],
    sparse_threshold=1.0
)

X_transformed = preprocessor_cluster.fit_transform(X_cluster)
print('Matriz transformada:', X_transformed.shape)
print('Formato:', type(X_transformed).__name__)


## **4. K-Means: selección del número de clusters**

Se evalúa `k` entre 2 y 8. Se utiliza `MiniBatchKMeans` para hacer viable el ajuste sobre el conjunto completo. La selección considera **inercia** (método del codo) y **silhouette** calculado sobre una muestra reproducible de hasta 4.000 registros. El valor de silhouette se interpreta junto con la separación visual y el tamaño de los grupos; no se elige automáticamente un `k` solo por maximizar una métrica.

In [ ]:
resultados_kmeans = []
ks = range(2, 9)
n_silhouette = min(4000, X_transformed.shape[0])

for k in ks:
    modelo = MiniBatchKMeans(
        n_clusters=k, random_state=42, batch_size=2048,
        n_init=10, max_iter=150
    )
    labels = modelo.fit_predict(X_transformed)
    sil = silhouette_score(
        X_transformed, labels,
        sample_size=n_silhouette, random_state=42
    )
    resultados_kmeans.append({
        'k': k,
        'inercia': modelo.inertia_,
        'silhouette_muestral': sil
    })

df_kmeans_eval = pd.DataFrame(resultados_kmeans)
display(df_kmeans_eval.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(df_kmeans_eval['k'], df_kmeans_eval['inercia'], marker='o')
axes[0].set_title('Método del codo — Inercia')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[1].plot(df_kmeans_eval['k'], df_kmeans_eval['silhouette_muestral'], marker='o')
axes[1].set_title('Coeficiente Silhouette (muestra)')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette')
plt.tight_layout()
plt.show()

## **5. Ajuste final y perfilado de K-Means**

Se toma como candidato el `k` con mejor silhouette entre las alternativas estudiadas, pero se documenta la distribución y se revisa que no existan grupos residuales demasiado pequeños. La gravedad no interviene en el ajuste: se incorpora únicamente en la etapa de interpretación.

In [ ]:
best_k = int(df_kmeans_eval.loc[df_kmeans_eval['silhouette_muestral'].idxmax(), 'k'])
modelo_kmeans = MiniBatchKMeans(
    n_clusters=best_k, random_state=42, batch_size=2048,
    n_init=20, max_iter=200
)
df['cluster_kmeans'] = modelo_kmeans.fit_predict(X_transformed)

distribucion_kmeans = (
    df['cluster_kmeans'].value_counts().sort_index()
      .rename_axis('cluster').to_frame('siniestros')
)
distribucion_kmeans['porcentaje'] = 100 * distribucion_kmeans['siniestros'] / len(df)
print(f'K seleccionado: {best_k}')
display(distribucion_kmeans.round(2))

# Perfil descriptivo: las columnas de resultado se usan solo después del clustering.
cols_perfil = ['hora_num','dia_semana_num','comuna_num','es_fin_de_semana',
               'involucra_moto','involucra_peaton','involucra_bicicleta',
               'involucra_transporte_publico','involucra_pesado']
perfil_kmeans = df.groupby('cluster_kmeans')[cols_perfil].mean().round(3)
perfil_kmeans['siniestros'] = df.groupby('cluster_kmeans').size()
if 'gravedad_siniestro' in df.columns:
    gravedad_limpia = df['gravedad_siniestro'].astype('string').str.upper().str.strip()
    df['es_critico_posthoc'] = gravedad_limpia.isin(['GRAVE','MORTAL']).astype('int8')
    perfil_kmeans['tasa_grave_mortal_posthoc'] = df.groupby('cluster_kmeans')['es_critico_posthoc'].mean().round(4)
display(perfil_kmeans)

## **6. DBSCAN: reducción dimensional y criterio de densidad**

DBSCAN puede ser costoso y sensible a la distancia en espacios dispersos de alta dimensión. Para hacerlo comparable y manejable, se reduce la matriz codificada con **TruncatedSVD** a 15 componentes y se estandarizan esas coordenadas. Se calcula la distancia al 10.º vecino más próximo y se exploran radios (`eps`) basados en los percentiles 85, 90 y 95 de esa distribución. Se informa explícitamente el porcentaje de ruido (`cluster = -1`).

In [ ]:
svd = TruncatedSVD(n_components=15, random_state=42)
X_reduced = svd.fit_transform(X_transformed)
X_reduced = StandardScaler().fit_transform(X_reduced)
print(f'Varianza explicada acumulada por SVD: {svd.explained_variance_ratio_.sum():.2%}')

rng = np.random.default_rng(42)
n_dbscan = min(10000, X_reduced.shape[0])
idx_dbscan = rng.choice(X_reduced.shape[0], size=n_dbscan, replace=False)
X_dbscan = X_reduced[idx_dbscan]

min_samples = 10
nn = NearestNeighbors(n_neighbors=min_samples, n_jobs=-1).fit(X_dbscan)
distancias, _ = nn.kneighbors(X_dbscan)
k_dist = np.sort(distancias[:, -1])
percentiles_eps = np.percentile(k_dist, [85, 90, 95])

plt.figure(figsize=(10, 4))
plt.plot(k_dist, linewidth=1)
for q, eps in zip([85,90,95], percentiles_eps):
    plt.axhline(eps, linestyle='--', label=f'P{q}: eps={eps:.3f}')
plt.title('Curva de distancia al 10.º vecino más próximo')
plt.xlabel('Observaciones ordenadas')
plt.ylabel('Distancia')
plt.legend()
plt.tight_layout()
plt.show()

## **7. Evaluación de DBSCAN y selección explícita**

Se comparan las tres distancias candidatas con `min_samples=10`. La silhouette se calcula solo sobre los puntos no considerados ruido y solo cuando existen al menos dos clusters válidos. La decisión no se fuerza: si las alternativas generan esencialmente ruido o un único grupo, se registra que DBSCAN no produjo una partición útil con esta representación y estos parámetros.

In [ ]:
resultados_dbscan = []
modelos_dbscan = {}
for q, eps in zip([85,90,95], percentiles_eps):
    modelo = DBSCAN(eps=float(eps), min_samples=min_samples, n_jobs=-1)
    labels = modelo.fit_predict(X_dbscan)
    no_ruido = labels != -1
    clusters_validos = set(labels[no_ruido])
    n_clusters = len(clusters_validos)
    noise_ratio = float((~no_ruido).mean())
    sil = np.nan
    if n_clusters >= 2 and no_ruido.sum() >= 100:
        sil = silhouette_score(
            X_dbscan[no_ruido], labels[no_ruido],
            sample_size=min(3000, int(no_ruido.sum())), random_state=42
        )
    resultados_dbscan.append({
        'percentil_eps': q, 'eps': float(eps),
        'clusters_sin_ruido': n_clusters,
        'ruido_pct': 100 * noise_ratio,
        'silhouette_sin_ruido': sil
    })
    modelos_dbscan[q] = (modelo, labels)

df_dbscan_eval = pd.DataFrame(resultados_dbscan)
display(df_dbscan_eval.round(4))
validos = df_dbscan_eval.dropna(subset=['silhouette_sin_ruido'])
if len(validos):
    mejor_q = int(validos.loc[validos['silhouette_sin_ruido'].idxmax(), 'percentil_eps'])
    modelo_dbscan, labels_dbscan = modelos_dbscan[mejor_q]
    print(f'DBSCAN candidato seleccionado: percentil {mejor_q}, eps={modelo_dbscan.eps:.4f}')
else:
    mejor_q = None
    labels_dbscan = None
    print('DBSCAN no encontró una partición con al menos dos clusters no-ruido en los radios explorados.')

## **8. Comparación e interpretación de resultados**

La elección del método considera: (1) calidad de separación, (2) proporción de observaciones asignadas, (3) cantidad y tamaño de grupos, (4) facilidad de describir perfiles útiles para prevención vial y (5) sensibilidad de los resultados a los parámetros. K-Means asigna todos los casos; DBSCAN puede dejar casos como ruido y detectar grupos de densidad no esféricos.

Los porcentajes de gravedad por cluster son **descriptivos y posteriores al ajuste**, no evidencian causalidad ni forman parte de la definición de los clusters.

In [ ]:
comparacion = pd.DataFrame([
    {
        'algoritmo':'MiniBatch K-Means',
        'clusters':best_k,
        'observaciones_asignadas':len(df),
        'observaciones_ruido':0,
        'silhouette_muestral':float(df_kmeans_eval.loc[df_kmeans_eval['k']==best_k,'silhouette_muestral'].iloc[0])
    }
])
if labels_dbscan is not None:
    comparacion = pd.concat([comparacion, pd.DataFrame([{
        'algoritmo':'DBSCAN (muestra)',
        'clusters':len(set(labels_dbscan)-{-1}),
        'observaciones_asignadas':int((labels_dbscan!=-1).sum()),
        'observaciones_ruido':int((labels_dbscan==-1).sum()),
        'silhouette_muestral':float(validos.loc[validos['percentil_eps']==mejor_q,'silhouette_sin_ruido'].iloc[0])
    }])], ignore_index=True)
display(comparacion)

# Resumen de tasas de gravedad por cluster K-Means (solo evaluación posterior)
if 'es_critico_posthoc' in df.columns:
    resumen_severidad = df.groupby('cluster_kmeans').agg(
        siniestros=('cluster_kmeans','size'),
        graves_mortales=('es_critico_posthoc','sum'),
        tasa_graves_mortales=('es_critico_posthoc','mean')
    ).sort_values('tasa_graves_mortales', ascending=False)
    display(resumen_severidad.round(4))

## **9. Conclusiones y decisiones metodológicas**

Completar esta sección luego de ejecutar el notebook y revisar las tablas generadas. La conclusión debe mencionar: el `k` elegido y por qué; los tamaños de los clusters; qué combinaciones temporales, territoriales y de actores caracterizan cada grupo; el resultado de DBSCAN y su proporción de ruido; y qué perfiles merecen investigación preventiva.

**Criterio de interpretación:** los grupos representan similitudes estadísticas entre registros, no categorías clínicas ni causas demostradas de siniestralidad. Una tasa mayor de casos graves/mortales dentro de un cluster es una asociación descriptiva y no una relación causal.